# CSVW cross-check — the base graphs against rdf-tabular

Each base graph is made by `cfa.rule.table_to_graph`, our implementation of CSVW minimal mode. Here the same table, with the same CSVW description, is converted by **rdf-tabular** (Ruby, the reference implementation of "Generating RDF from Tabular Data on the Web"), and the two graphs must be equal (same canonical N-Triples).

Needs Ruby with the gems `rdf-tabular` and `rdf-turtle` (`gem install rdf-tabular rdf-turtle`). For an xlsx source, the table is first written as CSV (RFC 4180) as `cfa.read` reads it; the CSVW `url` is pointed at that file.

In [ ]:
import csv
import json
import subprocess
import sys
import tempfile
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

from rdflib import Graph

from cfa import SCHEMA_DIR
from cfa.graph import canonical_lines
from cfa.pins import checked_source_file, table_file
from cfa.read import read_xlsx_table
from cfa.rule import load_module, module_source

RB = REPO_ROOT / 'checks' / 'csvw_rdf_tabular.rb'
print(subprocess.run(['gem', 'list', 'rdf-tabular'], capture_output=True, text=True, check=True).stdout)

## NSV Registry

In [ ]:
sv = load_module(SCHEMA_DIR / 'nsv_registry.yaml')
SOURCE = module_source(sv)
TABLE = 'Approved NSVs-alpha'
header_row = int(sv.get_class(TABLE).annotations['header_row'].value)
table = read_xlsx_table(checked_source_file(SOURCE, table_file(SOURCE, TABLE)), TABLE, skip_rows=header_row - 1)
meta = json.loads((SCHEMA_DIR / 'csvw' / f'{SOURCE}.csv-metadata.json').read_text(encoding='utf-8'))
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    with open(tmp / 'table.csv', 'w', encoding='utf-8', newline='') as f:
        w = csv.writer(f)
        w.writerow(table.columns)
        w.writerows(table.itertuples(index=False, name=None))
    meta['url'] = 'table.csv'
    (tmp / 'metadata.json').write_text(json.dumps(meta), encoding='utf-8')
    subprocess.run(['ruby', str(RB), 'metadata.json', 'rdf-tabular.nt'], cwd=tmp, check=True)
    theirs = canonical_lines(Graph().parse(tmp / 'rdf-tabular.nt', format='nt'))
ours = canonical_lines(Graph().parse(REPO_ROOT / 'base' / SOURCE / f'{SOURCE}.ttl'))
assert ours == theirs, 'rdf-tabular graph differs from the base graph'
print(SOURCE, ': rdf-tabular graph equal to the base graph,', len(ours), 'triples')